In [1]:
import pandas as pd

In [2]:
robin_df = pd.read_csv('robin_clean.csv')

In [3]:
# get smiles list 
smiles_list = robin_df['Smile'].tolist()

# Filter out smiles with unsupported atoms
# SMILES GA grammar supports: B, C, F, H, I, N, O, P, S, Cl, Br, Si, Se (and aromatic versions)
# Common unsupported atoms: As, Al, Ar, At, etc.

import re
from rdkit import Chem

# Define supported atoms in the SMILES grammar
supported_aliphatic = {'B', 'C', 'F', 'H', 'I', 'N', 'O', 'P', 'S', 'Cl', 'Br', 'Si', 'Se'}
supported_aromatic = {'b', 'c', 'n', 'o', 'p', 's', 'se'}
supported_atoms = supported_aliphatic | supported_aromatic

def is_smiles_supported(smiles):
    """Check if SMILES contains only supported atoms"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    
    # Check each atom in the molecule
    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        aromatic_symbol = symbol.lower()
        
        # Check if atom is supported (either aliphatic or aromatic form)
        if symbol not in supported_atoms and aromatic_symbol not in supported_atoms:
            return False
    
    return True

# Filter the smiles list
original_count = len(smiles_list)
smiles_list = [s for s in smiles_list if is_smiles_supported(s)]
filtered_count = len(smiles_list)

print(f"Original SMILES count: {original_count}")
print(f"Filtered SMILES count: {filtered_count}")
print(f"Removed {original_count - filtered_count} SMILES with unsupported atoms")


Original SMILES count: 24501
Filtered SMILES count: 24447
Removed 54 SMILES with unsupported atoms


[21:54:00] Conflicting single bond directions around double bond at index 27.
[21:54:00]   BondStereo set to STEREONONE and single bond directions set to NONE.


In [4]:
# save the filtered smiles list to a txt file
# Save to the input directory used by the optimizer
output_path = 'rnamigos2_minimal/inputs/ligands/robin_smiles.txt'
with open(output_path, 'w') as f:
    for smiles in smiles_list:
        f.write(smiles + '\n')

print(f"Saved {len(smiles_list)} filtered SMILES to {output_path}")


Saved 24447 filtered SMILES to rnamigos2_minimal/inputs/ligands/robin_smiles.txt


In [5]:
# Optional: Check which atoms caused filtering
from collections import Counter

def get_unsupported_atoms(smiles):
    """Return list of unsupported atoms in a SMILES string"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []
    
    unsupported = []
    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        aromatic_symbol = symbol.lower()
        
        if symbol not in supported_atoms and aromatic_symbol not in supported_atoms:
            unsupported.append(symbol)
    
    return unsupported

# Find all unsupported atoms across all original SMILES
all_unsupported = []
original_smiles_list = robin_df['Smile'].tolist()

for smiles in original_smiles_list:
    unsupported = get_unsupported_atoms(smiles)
    all_unsupported.extend(unsupported)

if all_unsupported:
    print("Unsupported atoms found:")
    atom_counts = Counter(all_unsupported)
    for atom, count in atom_counts.most_common():
        print(f"  {atom}: {count} occurrences")
else:
    print("No unsupported atoms found!")

Unsupported atoms found:
  As: 31 occurrences
  Na: 16 occurrences
  Pt: 5 occurrences
  Ge: 2 occurrences
  Sr: 2 occurrences
  K: 1 occurrences
  Au: 1 occurrences
  Mg: 1 occurrences
  Co: 1 occurrences


[21:54:24] Conflicting single bond directions around double bond at index 27.
[21:54:24]   BondStereo set to STEREONONE and single bond directions set to NONE.
